# Projeto de Amplificador CMOS de 2 Estágios
## EE640 — Eletrônica Básica II

**RA utilizado:** 258610  
**Parâmetros extraídos:**
- a=2, b=5, c=8, d=6, e=1, f=0
- ab=25, bc=58, cd=86, de=61, ef=10, df=60, ce=81

**Parâmetros do transistor:**
- $|V_t| = 0{,}5\,V$
- $k'_p = k'_n = 20\,\mu A/V^2$
- $\lambda = 0{,}01\,V^{-1}$
- $L = 1\,\mu m$
- $C_{bd} = 10\,fF$, $C_{gdo} = 1\,fF$, $C_{gso} = 1\,fF$

In [ ]:
import numpy as np

# ── Parâmetros do processo ──────────────────────────────────────────────────
Vt   = 0.5        # V  — tensão de threshold (magnitude)
kp   = 20e-6      # A/V² — k' de PMOS e NMOS
kn   = 20e-6
lam  = 0.01       # V⁻¹ — coeficiente de modulação de canal
L    = 1e-6       # m

# ── Alimentação ────────────────────────────────────────────────────────────
Vcc  =  7.5       # V
Vss  = -7.5       # V

# ── Parâmetros do RA ───────────────────────────────────────────────────────
RA   = 258610
digits = [int(d) for d in str(RA)]
a, b, c, d, e, f = digits

# Combinações usadas
df_val = int(f"{d}{f}")   # 60
de_val = int(f"{d}{e}")   # 61
cd_val = int(f"{c}{d}")   # 86
ce_val = int(f"{c}{e}")   # 81

print(f"RA: {RA}")
print(f"df={df_val}, de={de_val}, cd={cd_val}, ce={ce_val}")

---
## Questão 2 — Corrente de Referência e R1

### Circuito de polarização

M5 (PMOS) e M6 (NMOS) são diode-connected em série com R1, entre $V_{cc}$ e $V_{ss}$.  
A corrente que circula por esse ramo é a corrente de referência $I_{ref}$.

### Corrente de referência

$$I_{ref} = 15\,\mu A + df \times 10^{-7}$$

### Tensão sobre R1

Com $V_{GS5} = V_{GS6} = 1{,}5\,V$ escolhidos:

$$V_{G5} = V_{cc} - V_{SG5} = 7{,}5 - 1{,}5 = 6\,V$$
$$V_{G6} = V_{ss} + V_{GS6} = -7{,}5 + 1{,}5 = -6\,V$$
$$V_{R1} = V_{G5} - V_{G6} = 12\,V$$
$$R1 = \frac{V_{R1}}{I_{ref}}$$

### W/L de M5 e M6

Como M5 e M6 são diode-connected, $V_{DS} = V_{GS}$. Incluindo efeito Early:

$$I_{ref} = \frac{1}{2}k'\frac{W}{L}(V_{GS}-V_t)^2(1+\lambda V_{GS})$$

$$\frac{W}{L}_{M5,M6} = \frac{I_{ref}}{\frac{1}{2}k'(V_{GS}-V_t)^2(1+\lambda V_{GS})}$$

In [ ]:
# ── Q2: Corrente de referência e R1 ────────────────────────────────────────

Iref = 15e-6 + df_val * 1e-7
print(f"Iref = {Iref*1e6:.3f} µA")

# Tensão sobre R1
VGS_bias = 1.5          # V — escolha de projeto
Vov_bias = VGS_bias - Vt
VG5  = Vcc - VGS_bias   # gate de M5
VG6  = Vss + VGS_bias   # gate de M6
VR1  = VG5 - VG6
R1   = VR1 / Iref

print(f"VG5  = {VG5:.2f} V")
print(f"VG6  = {VG6:.2f} V")
print(f"VR1  = {VR1:.2f} V")
print(f"R1   = {R1/1e3:.2f} kΩ")

# W/L de M5 e M6 com efeito Early (Vds = Vgs, diode-connected)
WL_M5M6 = Iref / (0.5 * kn * Vov_bias**2 * (1 + lam * VGS_bias))
print(f"\nW/L (M5, M6) = {WL_M5M6:.4f}")

# Verificação de saturação: Vds = Vgs > Vov → sempre satisfeita
print(f"Vov (M5,M6)  = {Vov_bias:.3f} V")
print(f"Vds (M5,M6)  = {VGS_bias:.3f} V  ≥  Vov → saturado ✓")

---
## Questão 3 — 1º Estágio: Ganho $A_{v1} = 100 + de$

### Topologia

Par diferencial PMOS (M1, M2) com espelho de corrente NMOS (M3, M4) como carga ativa.  
M3/M4 espelham M5 com razão 1:1, então cada ramo do diferencial conduz $I_{ref}/2$.

### Ganho do 1º estágio

$$A_{v1} = g_{m1} \cdot (r_{o2} \| r_{o4})$$

Com $r_{o2} = r_{o4}$ (mesma corrente):

$$r_{o2}\|r_{o4} = \frac{1}{2\lambda I_{D1}}$$

### Transcondutância necessária

$$g_{m1,req} = \frac{A_{v1}}{r_{o2}\|r_{o4}}$$

### W/L de M1 e M2

$$\frac{W}{L}\bigg|_{M1,M2} = \frac{g_{m1}^2}{2k'_p I_{D1}}$$

In [ ]:
# ── Q3: 1º Estágio ─────────────────────────────────────────────────────────

Av1_target = 100 + de_val
print(f"Ganho alvo 1º estágio: Av1 = {Av1_target} V/V")

# Corrente em cada ramo do diferencial
ID1 = Iref / 2
print(f"\nID1 = ID2 = Iref/2 = {ID1*1e6:.3f} µA")

# Resistências de saída
ro2 = 1 / (lam * ID1)
ro4 = ro2
ro2_par_ro4 = ro2 / 2
print(f"ro2 = ro4 = {ro2/1e6:.4f} MΩ")
print(f"ro2 ‖ ro4 = {ro2_par_ro4/1e6:.4f} MΩ")

# Transcondutância necessária
gm1_req = Av1_target / ro2_par_ro4
print(f"\ngm1 necessário = {gm1_req*1e6:.4f} µA/V")

# W/L de M1 e M2
WL_M1M2 = gm1_req**2 / (2 * kp * ID1)
print(f"W/L (M1, M2) = {WL_M1M2:.4f}")

# Verificação
gm1 = np.sqrt(2 * kp * WL_M1M2 * ID1)
Av1 = gm1 * ro2_par_ro4
Vov1 = 2 * ID1 / gm1
VGS1 = Vt + Vov1

print(f"\n── Verificação ──")
print(f"gm1   = {gm1*1e6:.4f} µA/V")
print(f"Av1   = {Av1:.2f} V/V  (alvo: {Av1_target})")
print(f"Vov1  = {Vov1*1e3:.2f} mV")
print(f"VGS1  = {VGS1:.4f} V")

---
## Questão 4 — 2º Estágio: Ganho $A_{v2} = 100 + cd$

### Topologia

M8 (PMOS) como transistor de ganho, M9 (NMOS) como carga espelhada de M7.  
M7 e M9 espelham M6 com razão 1:1, portanto $I_{D8} = I_{ref}$.

### Ganho do 2º estágio

$$A_{v2} = g_{m8} \cdot (r_{o8} \| r_{o9}) = g_{m8} \cdot \frac{1}{2\lambda I_{D8}}$$

### W/L de M8

$$g_{m8,req} = \frac{A_{v2}}{r_{o8}\|r_{o9}}$$

$$\frac{W}{L}\bigg|_{M8} = \frac{g_{m8}^2}{2k'_p I_{D8}}$$

M7 e M9 espelham M6, portanto:

$$\frac{W}{L}_{M7,M9} = \frac{W}{L}_{M6}$$

In [ ]:
# ── Q4: 2º Estágio ─────────────────────────────────────────────────────────

Av2_target = 100 + cd_val
print(f"Ganho alvo 2º estágio: Av2 = {Av2_target} V/V")

# Corrente no 2º estágio (espelho 1:1 com M6)
ID8 = Iref
print(f"\nID8 = Iref = {ID8*1e6:.3f} µA")

# Resistências de saída
ro8 = 1 / (lam * ID8)
ro9 = ro8
ro8_par_ro9 = ro8 / 2
print(f"ro8 = ro9 = {ro8/1e6:.4f} MΩ")
print(f"ro8 ‖ ro9 = {ro8_par_ro9/1e6:.4f} MΩ")

# Transcondutância necessária
gm8_req = Av2_target / ro8_par_ro9
print(f"\ngm8 necessário = {gm8_req*1e6:.4f} µA/V")

# W/L de M8
WL_M8 = gm8_req**2 / (2 * kp * ID8)
WL_M7M9 = WL_M5M6   # espelho 1:1 com M6
print(f"W/L (M8)     = {WL_M8:.4f}")
print(f"W/L (M7, M9) = {WL_M7M9:.4f}  (igual a M6)")

# Verificação
gm8 = np.sqrt(2 * kp * WL_M8 * ID8)
Av2 = gm8 * ro8_par_ro9
Vov8 = 2 * ID8 / gm8
Vov9 = np.sqrt(2 * ID8 / (kn * WL_M7M9))
VGS9 = Vt + Vov9

print(f"\n── Verificação ──")
print(f"gm8   = {gm8*1e6:.4f} µA/V")
print(f"Av2   = {Av2:.2f} V/V  (alvo: {Av2_target})")
print(f"Vov8  = {Vov8*1e3:.2f} mV")
print(f"Vov9  = {Vov9*1e3:.2f} mV")
print(f"VGS9  = {VGS9:.4f} V  (esperado ≈ 1.5 V ✓)")

# Swing de saída
Vout_max =  Vcc - Vov8
Vout_min =  Vss + Vov9
print(f"\n── Swing de saída (Q7 preview) ──")
print(f"Vout_max = Vcc - Vov8 = {Vout_max:.3f} V")
print(f"Vout_min = Vss + Vov9 = {Vout_min:.3f} V")

---
## Questão 5 — M3 e M4: Redução de Offset Sistemático

### Origem do offset

O offset sistemático surge porque a corrente espelhada por M4 para o nó de saída do 1º estágio precisa ser exatamente igual à corrente que o 2º estágio consome. A condição geral é:

$$\frac{(W/L)_{M8}}{(W/L)_{M4}} = 2 \cdot \frac{(W/L)_{M9}}{(W/L)_{M7}}$$

O fator 2 vem do fato de M4 conduzir $I_{ref}/2$ enquanto M9 conduz $I_{ref}$.

### Cálculo de W/L de M3 e M4

Isolando $(W/L)_{M4}$:

$$\frac{W}{L}\bigg|_{M3,M4} = \frac{(W/L)_{M8}}{2 \cdot \dfrac{(W/L)_{M9}}{(W/L)_{M7}}}$$

In [ ]:
# ── Q5: Offset ─────────────────────────────────────────────────────────────

# Condição de offset zero:
# WL_M8 / WL_M4 = 2 * (WL_M9 / WL_M7)
# Como WL_M9 = WL_M7 (espelho 1:1):
razao_espelho = WL_M7M9 / WL_M7M9   # = 1
WL_M3M4 = WL_M8 / (2 * razao_espelho)

print(f"Condição de offset zero:")
print(f"  WL_M8 / WL_M4 = 2 × (WL_M9/WL_M7)")
print(f"  {WL_M8:.4f} / WL_M4 = 2 × {razao_espelho:.1f}")
print(f"\nW/L (M3, M4) = {WL_M3M4:.4f}")

# Verificação da condição
lhs = WL_M8 / WL_M3M4
rhs = 2 * (WL_M7M9 / WL_M7M9)
print(f"\n── Verificação ──")
print(f"WL_M8/WL_M4 = {lhs:.4f}")
print(f"2×WL_M9/WL_M7 = {rhs:.4f}")
print(f"Offset zero: {np.isclose(lhs, rhs)} ✓")

# Overdrive de M3/M4
ID3 = ID1
Vov3 = np.sqrt(2 * ID3 / (kn * WL_M3M4))
VGS3 = Vt + Vov3
print(f"\nVov3 = {Vov3*1e3:.2f} mV")
print(f"VGS3 = {VGS3:.4f} V")
print(f"Saturação M3 (diode-conn): Vds={VGS3:.3f} V ≥ Vov={Vov3:.3f} V ✓")

---
## Questão 6 — Compensação de Miller: $C_f$, Polos, Zeros e Margens

### Por que $\omega_T = g_{m1}/C_f$?

O GBW do OTA é o produto do ganho DC pelo polo dominante. Os termos de $R_{out}$ e $A_{v2}$ se cancelam:

$$GBW = A_{v0} \cdot \omega_{p1} = (g_{m1} R_{out1} A_{v2}) \cdot \frac{1}{C_f A_{v2} R_{out1}} = \frac{g_{m1}}{C_f}$$

### Polo dominante (efeito Miller)

$$\omega_{p1} = \frac{1}{C_f \cdot A_{v2} \cdot (r_{o2}\|r_{o4})}$$

### Polo não-dominante e zero RHP

$$\omega_{p2} = \omega_z = \frac{g_{m8}}{C_f}$$

O zero está no RHP (semiplano direito) pois o caminho de feedforward por $C_f$ não inverte o sinal. Zero RHP **adiciona** atraso de fase — mesmo efeito de um polo.

### Margem de fase

Como $\omega_{p2} = \omega_z$ (com sinais opostos de fase):

$$PM = 90° - 2\arctan\!\left(\frac{\omega_T}{\omega_{p2}}\right)$$

Para estabilidade adequada: $PM > 45°$ (equivale a $\zeta > 0{,}5$, sobressinal < 16%).

In [ ]:
# ── Q6: Compensação e frequência ───────────────────────────────────────────

fT = 20e6 + ce_val * 1e5
wT = 2 * np.pi * fT
print(f"fT alvo = {fT/1e6:.2f} MHz")
print(f"ωT      = {wT/1e6:.2f} Mrad/s")

# Capacitor de compensação
Cf = gm1 / wT
print(f"\nCf = gm1/ωT = {Cf*1e15:.2f} fF")

# Polo dominante
wp1 = 1 / (Cf * Av2 * ro2_par_ro4)
fp1 = wp1 / (2 * np.pi)
print(f"\nPolo dominante:")
print(f"  ωp1 = {wp1:.2f} rad/s")
print(f"  fp1 = {fp1:.2f} Hz")

# Polo não-dominante e zero RHP
wp2 = gm8 / Cf
fp2 = wp2 / (2 * np.pi)
wz  = wp2   # mesmo valor, RHP
fz  = fp2
print(f"\nPolo não-dominante e zero RHP:")
print(f"  ωp2 = ωz = {wp2/1e6:.2f} Mrad/s")
print(f"  fp2 = fz  = {fp2/1e6:.2f} MHz")

# Verificação de estabilidade
ratio = wp2 / wT
print(f"\nωp2/ωT = {ratio:.3f}  (deve ser > 1 ✓)")

# Margem de fase
PM = 90 - 2 * np.degrees(np.arctan(wT / wp2))
print(f"\nMargem de fase:")
print(f"  PM = 90° - 2×arctan(ωT/ωp2)")
print(f"  PM = 90° - 2×arctan({wT/wp2:.4f})")
print(f"  PM = {PM:.2f}°")
if PM >= 45:
    print(f"  PM ≥ 45° ✓")
else:
    print(f"  PM < 45° ⚠ — considerar resistor de zero Rz = 1/gm8 = {1/gm8/1e3:.2f} kΩ")

# Margem de ganho
print(f"\nMargem de ganho: GM → ∞ (sem CL, fase não cruza -180°)")

# Ganho total
Av0 = Av1 * Av2
Av0_dB = 20 * np.log10(Av0)
print(f"\nGanho total:")
print(f"  Av0 = Av1 × Av2 = {Av1:.2f} × {Av2:.2f} = {Av0:.0f} V/V")
print(f"  Av0 = {Av0_dB:.2f} dB")

# Verificação GBW
GBW = Av0 * fp1
print(f"\nVerificação GBW:")
print(f"  GBW = Av0 × fp1 = {GBW/1e6:.3f} MHz  (alvo: {fT/1e6:.2f} MHz)")

# Sugestão de Rz para melhorar PM
Rz = 1 / gm8
print(f"\nResistor de zero (opcional, para PM→90°):")
print(f"  Rz = 1/gm8 = {Rz/1e3:.2f} kΩ")

---
## Resumo Final do Projeto

In [ ]:
# ── Resumo ─────────────────────────────────────────────────────────────────

print("═" * 55)
print("        RESUMO DO PROJETO — OTA CMOS 2 ESTÁGIOS")
print("═" * 55)

print("\n── Parâmetros de projeto ──────────────────────────────")
print(f"  Iref          = {Iref*1e6:.3f} µA")
print(f"  R1            = {R1/1e3:.2f} kΩ")
print(f"  Av1           = {Av1:.2f} V/V  (alvo: {Av1_target})")
print(f"  Av2           = {Av2:.2f} V/V  (alvo: {Av2_target})")
print(f"  Av0           = {Av0:.0f} V/V  ({Av0_dB:.2f} dB)")
print(f"  Cf            = {Cf*1e15:.2f} fF")
print(f"  fT            = {fT/1e6:.2f} MHz")
print(f"  fp1           = {fp1:.2f} Hz")
print(f"  fp2 = fz      = {fp2/1e6:.2f} MHz")
print(f"  PM            = {PM:.2f}°")

print("\n── Transistores ───────────────────────────────────────")
print(f"  {'Transistor':<12} {'Tipo':<6} {'W/L':>8}  {'ID (µA)':>10}  {'Vov (mV)':>10}")
print(f"  {'-'*52}")

transistors = [
    ("M5",    "PMOS", WL_M5M6,  Iref,  Vov_bias),
    ("M6",    "NMOS", WL_M5M6,  Iref,  Vov_bias),
    ("M1,M2", "PMOS", WL_M1M2,  ID1,   Vov1),
    ("M3,M4", "NMOS", WL_M3M4,  ID3,   Vov3),
    ("M7,M9", "NMOS", WL_M7M9,  ID8,   Vov9),
    ("M8",    "PMOS", WL_M8,    ID8,   Vov8),
]

for name, tipo, wl, id_, vov in transistors:
    print(f"  {name:<12} {tipo:<6} {wl:>8.4f}  {id_*1e6:>10.3f}  {vov*1e3:>10.2f}")

print("═" * 55)

---
## Modelos SPICE

```spice
.model MbreakN-X NMOS
+ VTO=0.5
+ L=1u
+ kp=20u
+ lambda=0.01
+ Cbd=10f
+ Cgdo=1f
+ Cgso=1f

.model MbreakP-X PMOS
+ VTO=-0.5
+ L=1u
+ kp=20u
+ lambda=0.01
+ Cbd=10f
+ Cgdo=1f
+ Cgso=1f
```

### Netlist do circuito de referência (Q2)

```spice
* Polarização
M5  nbias  nbias  vcc  vcc  MbreakP-X  W=2.069u  L=1u
M6  nbias2 nbias2 vss  vss  MbreakN-X  W=2.069u  L=1u
R1  nbias  nbias2  571.4k
```